In [1]:
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import shap

from modeling.pipelines.modeling.nodes import inference

/home/zaccosenza/code/project-311/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_PATH = "../data/dev/02_reporting/model.pkl"
MODELING_DATA_PATH = "../data/dev/02_reporting/modeling_data.parquet"

TARGET_COL = "tgt_calls"
FEATURE_COLS = [
    "ft_week_of_year", "ft_lag_1", "ft_lag_2", "ft_event_count",
    "ft_lag1_temp_max", "ft_lag1_temp_min", "ft_lag1_had_rain", "ft_lag1_had_snow",
    "ft_pred_temp_max", "ft_pred_temp_min", "ft_pred_had_rain", "ft_pred_had_snow",
    "ft_board_key",
]
# ft_board_key excluded from "reasons" below on purpose — it being the top SHAP
# contributor just says "this board has a high baseline", which is true every week
# and drowns out the actually-interesting, week-varying drivers (lag/events/weather).
REASON_FEATURES = [f for f in FEATURE_COLS if f != "ft_board_key"]

N_WEEKS = 8         # most recent weeks to analyze
TOP_DISTRICTS = 5   # top predicted districts per week
TOP_REASONS = 3     # top-N SHAP-contributing features per district

In [3]:
with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

modeling_data = pd.read_parquet(MODELING_DATA_PATH)
pred_col = f"pred_{TARGET_COL}"
scored = inference(model, modeling_data, FEATURE_COLS, TARGET_COL)

recent_weeks = sorted(scored["week_start"].unique())[-N_WEEKS:]
scored = scored[scored["week_start"].isin(recent_weeks)].reset_index(drop=True)
print(f"{len(scored):,} rows across {len(recent_weeks)} weeks: {recent_weeks[0]} .. {recent_weeks[-1]}")

624 rows across 8 weeks: 2026-06-01 .. 2026-07-20


In [4]:
top_districts = (
    scored.sort_values(pred_col, ascending=False)
    .groupby("week_start")
    .head(TOP_DISTRICTS)
    .reset_index(drop=True)
)
print(f"{len(top_districts):,} (week, district) rows in the top {TOP_DISTRICTS} per week")
top_districts[["board_key", "week_start", pred_col]].head(10)

40 (week, district) rows in the top 5 per week


,board_key,week_start,pred_tgt_calls
0,12 MANHATTAN,2026-06-29,2937.238770
1,12 MANHATTAN,2026-07-06,2672.291016
2,04 BRONX,2026-06-29,2581.131348
3,05 BRONX,2026-07-06,2233.006104
4,04 BRONX,2026-07-06,2215.328125
5,12 MANHATTAN,2026-07-13,2207.310791
6,07 BRONX,2026-06-29,2151.970703
7,05 BROOKLYN,2026-06-29,2136.316650
8,12 MANHATTAN,2026-06-22,2091.858398
9,12 MANHATTAN,2026-06-01,2078.354736


In [5]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(top_districts[FEATURE_COLS], check_additivity=False)
shap_df = pd.DataFrame(shap_values, columns=FEATURE_COLS, index=top_districts.index)


def top_reasons(row, k=TOP_REASONS):
    return row[REASON_FEATURES].abs().sort_values(ascending=False).head(k).index.tolist()


top_districts["top_reasons"] = shap_df.apply(top_reasons, axis=1)
top_districts[["board_key", "week_start", pred_col, "top_reasons"]].head(10)

,board_key,week_start,pred_tgt_calls,top_reasons
0,12 MANHATTAN,2026-06-29,2937.238770,"[ft_lag_1, ft_lag_2, ft_pred_temp_max]"
1,12 MANHATTAN,2026-07-06,2672.291016,"[ft_lag_1, ft_lag_2, ft_lag1_temp_max]"
2,04 BRONX,2026-06-29,2581.131348,"[ft_lag_1, ft_lag_2, ft_pred_temp_max]"
3,05 BRONX,2026-07-06,2233.006104,"[ft_lag_1, ft_lag_2, ft_lag1_temp_max]"
4,04 BRONX,2026-07-06,2215.328125,"[ft_lag_1, ft_lag_2, ft_lag1_temp_max]"
5,12 MANHATTAN,2026-07-13,2207.310791,"[ft_lag_1, ft_lag_2, ft_pred_temp_max]"
6,07 BRONX,2026-06-29,2151.970703,"[ft_lag_1, ft_lag_2, ft_pred_temp_max]"
7,05 BROOKLYN,2026-06-29,2136.316650,"[ft_lag_1, ft_lag_2, ft_pred_temp_max]"
8,12 MANHATTAN,2026-06-22,2091.858398,"[ft_lag_1, ft_lag_2, ft_pred_temp_max]"
9,12 MANHATTAN,2026-06-01,2078.354736,"[ft_lag_1, ft_lag_2, ft_pred_temp_max]"


In [6]:
# Each of the top-K districts contributes its top-N reasons as separate votes — a week
# where one reason dominates every district's prediction shows up as a single-color bar,
# a week with varied drivers shows up as many colors.
reason_counts = (
    top_districts.explode("top_reasons")
    .groupby(["week_start", "top_reasons"])
    .size()
    .unstack(fill_value=0)
)
reason_counts

top_reasons,ft_lag1_temp_max,ft_lag_1,ft_lag_2,ft_pred_temp_max,ft_pred_temp_min
week_start,,,,,
2026-06-01,0,5,5,5,0
2026-06-08,0,5,5,5,0
2026-06-15,0,5,5,5,0
2026-06-22,0,5,5,5,0
2026-06-29,0,5,5,5,0
2026-07-06,3,5,5,2,0
2026-07-13,0,5,5,5,0
2026-07-20,0,5,5,0,5


In [8]:
fig, ax = plt.subplots(figsize=(12, 6))
reason_counts.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_xlabel("week")
ax.set_ylabel(f"# of top-{TOP_DISTRICTS} districts citing this reason (top-{TOP_REASONS} each)")
ax.set_title(f"Diversity of top SHAP reasons behind the top-{TOP_DISTRICTS} predicted districts, per week")
ax.legend(title="reason (feature)", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

/tmp/ipykernel_8347/3013181316.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
